# 🏡 San Antonio Automated Apartment & Townhome Hunter (v5 — Clean Frontend Dashboard)
### Welcome to Your Apartment-Hunting Control Center
To keep this notebook clean and easy to customize, **all complex algorithms, KD-Tree indexers, scoring criteria, and vision models are safely tucked away in backend modules (`src/`)**. 

Use the dashboard cells below to adjust your search radius, budget, weights, and persistent taste memory!

In [ ]:
# @title 🔌 Setup: Install Dependencies & Mount Google Drive Cache
!pip install -q homeharvest osmnx geopandas folium shapely scipy requests pandas pyyaml transformers torch pillow scikit-learn

import sys
import os
if 'src' not in os.listdir('.'):
    print('Cloning or setting up repo path...')

# Set to True to persist your OSM cache, CLIP embedding cache, and taste memory on Google Drive!
USE_GOOGLE_DRIVE_MEMORY = True

if USE_GOOGLE_DRIVE_MEMORY:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/LoveWhereYouLive_cache'
    PREF_FILE = '/content/drive/MyDrive/LoveWhereYouLive_preferences.json'
else:
    CACHE_DIR = 'cache'
    PREF_FILE = 'data/preferences.json'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('✅ Frontend initialized! Google Drive Cache Directory:', CACHE_DIR)

In [ ]:
# @title 🎛️ Customize Your Search Filters, Cutoffs & Criteria Weights
import yaml

# Load config or set custom overrides directly in this dashboard UI cell:
CONFIG = {
    'workplace': {'name': 'Southwest Research Institute (SwRI)', 'coords': (29.4485, -98.6105)},
    'filters': {
        'location': 'San Antonio, TX',
        'min_rent': 1200,
        'max_rent': 2200,
        'min_beds': 2,
        'min_baths': 1.5,
        'past_days': 30
    },
    'cutoffs': {
        'require_in_unit_laundry_and_dishwasher': True,
        'max_swri_commute_mins': 28.0,
        'max_grocery_dist_m': 2400.0,
        'max_park_trail_dist_m': 1600.0
    },
    # 🎚️ Adjust your priority weights here (must sum to 1.0):
    'weights': {
        'commute_score': 0.16,
        'grocery_score': 0.11,
        'park_trail_score': 0.11,
        'pollution_highway_score': 0.07,
        'modern_interior_score': 0.10,
        'garage_bike_score': 0.07,
        'costco_score': 0.07,
        'water_softener_score': 0.05,
        'pet_flooring_score': 0.05,
        'effective_rent_score': 0.05,
        'noise_buffer_score': 0.05,
        'recency_score': 0.06
    }
}
print('✅ Dashboard filters and weights configured!')

In [ ]:
# @title 🚀 Run Pipeline (Loads Cached OSM Trees & Scrapes Listings in Seconds)
from src.pipeline import run_pipeline

# Executes end-to-end backend pipeline using Google Drive cache and outputs ranked CSV + Visual Audit HTML
scored_df = run_pipeline(
    config_dict=CONFIG,
    cache_dir=CACHE_DIR,
    pref_file=PREF_FILE,
    run_local_clip=True,
    clip_max_candidates=25
)

scored_df[['listing_status', 'composite_score', 'formatted_address', 'list_price', 'property_age_years', 'modern_interior_score', 'swri_rush_hour_mins', 'nearest_grocery']].head(15)

In [ ]:
# @title ⭐ Manage Persistent Taste Memory (Favorites & Long-Term Blacklists)
from src.preference_memory import record_feedback, load_preferences

# 1. Add a favorite property to your shortlist (pins to outputs/favorites_shortlist.csv):
# record_feedback(address='123 Example St, San Antonio, TX', liked=True, reason='Great kitchen & garage', path=PREF_FILE)

# 2. Permanently blacklist an entire apartment complex name across all future searches:
# record_feedback(liked=False, blacklist_complex_name='Villas at Example', path=PREF_FILE)

prefs = load_preferences(PREF_FILE)
print('⭐ Current Favorites Shortlist:', prefs['taste_profile']['liked_property_addresses'])
print('🚫 Blacklisted Complexes:', prefs['taste_profile']['blacklisted_complex_keywords'])

In [ ]:
# @title 🔍 Single-Link / Address Sanity Check Inspector
from src.inspector import inspect_and_explain_listing
from src.criteria import ACTIVE_CRITERIA
from src.criteria.vision_grader import score_photos_with_local_clip
from src.preference_memory import load_preferences, record_feedback

# Paste any Realtor.com, Zillow, or Redfin link (or street address) below:
TARGET_URL_OR_ADDRESS = "" 

if TARGET_URL_OR_ADDRESS.strip():
    prefs = load_preferences(PREF_FILE)
    waterfall_df = inspect_and_explain_listing(
        TARGET_URL_OR_ADDRESS,
        raw_df=raw if 'raw' in locals() else None,
        ctx={'config': CONFIG, 'trees': TREES if 'TREES' in locals() else {}},
        active_criteria=ACTIVE_CRITERIA,
        vision_fn=score_photos_with_local_clip,
        liked_urls=prefs['taste_profile'].get('liked_kitchen_photo_urls', []),
        disliked_urls=prefs['taste_profile'].get('disliked_kitchen_photo_urls', []),
        clip_cache_path=os.path.join(CACHE_DIR, 'clip_photo_cache.pkl'),
        add_to_favorites=False,
        add_to_blacklist=False,
        pref_file=PREF_FILE,
        record_feedback_fn=record_feedback
    )
    display(waterfall_df)
else:
    print('💡 Paste any listing URL or street address into `TARGET_URL_OR_ADDRESS` above and run this cell!')